# TempoVul: Stage 4 Correction and Corrected Analysis

Reproduces the Stage 4 artifact correction and every corrected table and figure reported in the paper (Tables 6 through 13, Appendices A through C, Figures 2, 3, 4, and 5).

**Background.** Post-submission validation identified that `stage4_testing` contained a literal placeholder string in place of real static analysis findings across all 400 samples, rather than the intended per-tool findings summary. This notebook first rebuilds the Stage 4 artifact correctly, then re-runs the affected evaluations, then reproduces the corrected statistical analysis. See the paper's Discussion (Section 5.3) for the full account and root cause.

**Inputs required:** `tempovul_with_artifacts_final.csv`, per-tool Stage 3 static analysis outputs, `llm_master.csv` (Stages 1 through 3, unaffected by the defect and reused as-is).

## 1. Rebuild the Stage 4 artifact

Findings are placed before the code, rather than after, so they are preserved under the downstream 2,000-character model context truncation regardless of code length.

In [ ]:
import pandas as pd

df = pd.read_csv('tempovul_with_artifacts_final.csv')
assert len(df) == 400

tools = ['flawfinder', 'cppcheck', 'semgrep', 'infer', 'codeql']
tool_dfs = {tool: pd.read_csv(f'static_analysis_results/{tool}_complete.csv').set_index('sample_id') for tool in tools}

def build_findings_summary(idx):
    lines = []
    any_findings = False
    for tool in tools:
        t = tool_dfs[tool]
        if idx not in t.index:
            lines.append(f"- {tool}: [no result available]")
            continue
        row = t.loc[idx]
        pred, cwe, n = row[f'{tool}_pred'], row[f'{tool}_cwe'], row[f'{tool}_findings']
        if pred == 1:
            any_findings = True
            cwe_display = cwe if pd.notna(cwe) else 'unspecified CWE'
            lines.append(f"- {tool}: {int(n)} finding(s), suspected {cwe_display}")
        else:
            lines.append(f"- {tool}: no findings")
    header = "STATIC ANALYSIS FINDINGS:" if any_findings else "STATIC ANALYSIS FINDINGS (no tool flagged this sample):"
    return header + "\n" + "\n".join(lines)

new_stage4 = []
for idx, row in df.iterrows():
    findings = build_findings_summary(idx)
    code_text = row['stage3_implementation']
    new_stage4.append(f"{findings}\n\n--- SOURCE CODE ---\n{code_text}")

df['stage4_testing'] = new_stage4

leftover = df['stage4_testing'].str.contains(r'\[Static analysis placeholder\]', regex=True).sum()
assert leftover == 0, f"Placeholder still present in {leftover} rows"

df.to_csv('tempovul_with_artifacts_v2.csv', index=False)
print(f"Saved tempovul_with_artifacts_v2.csv, placeholder remaining: {leftover}")

## 2. Re-run Stage 4 evaluation for all 5 models

Only Stage 4 is re-run (2,000 evaluations). Stages 1 through 3 used correct artifacts throughout and are not affected.

In [ ]:
# Run per model on the Argonne Leadership Computing Facility Polaris cluster.
# See src/utils/evaluate_stage4_only.py and the corresponding PBS job scripts
# for the full batch evaluation code. Output: tempovul_results_stage4_fix/{model}_stage4fix_complete.csv,
# one file per model, 400 rows each.

## 3. Merge corrected Stage 4 predictions into the original 4-stage results

In [ ]:
llm_models = ['deepseek', 'codellama', 'starcoder2', 'mistral', 'wizardcoder']
stage4_dfs = {m: pd.read_csv(f'tempovul_results_stage4_fix/{m}_stage4fix_complete.csv') for m in llm_models}

merged = stage4_dfs['deepseek'][['sample_id', 'ground_truth', 'category', 'cwe']].copy()
for m in llm_models:
    merged = merged.merge(stage4_dfs[m][['sample_id', f'{m}_pred']], on='sample_id', how='outer')

llm_master = pd.read_csv('llm_master.csv')
stages_123 = llm_master[llm_master['stage_name'] != 'stage4'].copy()

stage4_corrected = merged.copy()
stage4_corrected['stage'] = 4
stage4_corrected['stage_name'] = 'stage4'
stage4_corrected = stage4_corrected[
    ['sample_id', 'stage', 'stage_name', 'ground_truth', 'category', 'cwe',
     'codellama_pred', 'starcoder2_pred', 'deepseek_pred', 'mistral_pred', 'wizardcoder_pred']
]

llm_master_v2 = pd.concat([stages_123, stage4_corrected], ignore_index=True)
llm_master_v2 = llm_master_v2.sort_values(['sample_id', 'stage']).reset_index(drop=True)

llm_master_v2.to_csv('llm_master_v2.csv', index=False)
print(f"Saved llm_master_v2.csv: {llm_master_v2.shape}")

## 4. Recompute Table 6 (LLM classification performance, all 4 stages)

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix

stages = ['stage1', 'stage2', 'stage3', 'stage4']
rows = []
for model in llm_models:
    for stage in stages:
        sub = llm_master_v2[llm_master_v2['stage_name'] == stage]
        y_true, y_pred = sub['ground_truth'], sub[f'{model}_pred']
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
        rows.append({'model': model, 'stage': stage, 'acc': round(acc,3), 'prec': round(prec,3),
                      'rec': round(rec,3), 'f1': round(f1,3), 'fpr': round(fpr,3), 'fnr': round(fnr,3)})

table6 = pd.DataFrame(rows)
print(table6.to_string(index=False))
table6.to_csv('table6_llm_classification_corrected.csv', index=False)

## 5. Recompute Table 7 (static tool classification)

Static tools are code-only and confirmed stage-invariant between Stage 3 and Stage 4 for Flawfinder, Semgrep, Infer, and CodeQL. cppcheck's original Stage 4 run encountered a syntax-error parsing artifact from the (now-fixed) placeholder text and is excluded; its Stage 3 value is reported as the single, canonical measurement for all five tools.

In [ ]:
static_master = pd.read_csv('static_master.csv')
static_tools = ['flawfinder', 'cppcheck', 'semgrep', 'infer', 'codeql']
stage3_static = static_master[static_master['stage'] == 3]

rows = []
for tool in static_tools:
    y_true, y_pred = stage3_static['ground_truth'], stage3_static[f'{tool}_pred']
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    rows.append({'tool': tool, 'acc': round(acc,3), 'prec': round(prec,3), 'rec': round(rec,3),
                  'f1': round(f1,3), 'fpr': round(fpr,3), 'fnr': round(fnr,3)})

table7 = pd.DataFrame(rows)
print(table7.to_string(index=False))
table7.to_csv('table7_static_classification_corrected.csv', index=False)

## 6. Recompute Table 8 (VD-S metric)

In [ ]:
vds_rows = []
for model in llm_models:
    for stage in stages:
        sub = llm_master_v2[llm_master_v2['stage_name'] == stage]
        y_true, y_pred = sub['ground_truth'], sub[f'{model}_pred']
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
        eligible = fpr <= 0.05
        vds = round(fnr, 3) if eligible and not np.isnan(fnr) else 'N/A'
        vds_rows.append({'model': model, 'stage': stage, 'fpr': round(fpr,3), 'vds': vds,
                          'eligible': 'Yes' if (eligible and vds != 'N/A') else 'No'})

vds_df = pd.DataFrame(vds_rows)
print(vds_df.to_string(index=False))
vds_df.to_csv('table8_vds_corrected.csv', index=False)

## 7. Survival analysis: time to detection, median TTD, EDR (Table 10)

In [ ]:
!pip install lifelines --quiet
from lifelines.statistics import logrank_test

def compute_ttd(df, model):
    vuln = df[df['ground_truth'] == 1].copy()
    times, events = [], []
    for sample_id in vuln['sample_id'].unique():
        sample_rows = df[df['sample_id'] == sample_id].sort_values('stage')
        detected_stage = None
        for _, row in sample_rows.iterrows():
            if row[f'{model}_pred'] == 1:
                detected_stage = row['stage']
                break
        if detected_stage is not None:
            times.append(detected_stage); events.append(1)
        else:
            times.append(4); events.append(0)
    return np.array(times), np.array(events)

ttd_results = {m: compute_ttd(llm_master_v2, m) for m in llm_models}

for model in llm_models:
    times, events = ttd_results[model]
    median_stage = np.median(times[events == 1]) if events.sum() > 0 else None
    n_censored = (events == 0).sum()
    edr = ((times <= 2) & (events == 1)).sum() / len(times)
    print(f"{model:12s} median_TTD={median_stage}  n_censored={n_censored} ({n_censored/len(events)*100:.1f}%)  EDR={edr:.3f}")

## 8. Log-rank tests: pairwise between LLMs, and best LLM versus best static tool (Table 11, Appendix A)

In [ ]:
from statsmodels.stats.multitest import multipletests
from itertools import combinations

logrank_rows = []
for m1, m2 in combinations(llm_models, 2):
    t1, e1 = ttd_results[m1]
    t2, e2 = ttd_results[m2]
    result = logrank_test(t1, t2, event_observed_A=e1, event_observed_B=e2)
    logrank_rows.append({'model_a': m1, 'model_b': m2, 'chi2': round(result.test_statistic,3), 'p': result.p_value})

logrank_df = pd.DataFrame(logrank_rows)
reject_holm, p_holm, _, _ = multipletests(logrank_df['p'], alpha=0.05, method='holm')
reject_bonf, p_bonf, _, _ = multipletests(logrank_df['p'], alpha=0.05, method='bonferroni')
logrank_df['p_holm'], logrank_df['sig_holm'] = p_holm, reject_holm
logrank_df['p_bonf'], logrank_df['sig_bonf'] = p_bonf, reject_bonf
print(logrank_df.to_string(index=False))
logrank_df.to_csv('appendixA_logrank_pairwise_llms_corrected.csv', index=False)

In [ ]:
# Best LLM (Mistral) vs best static tool (cppcheck), Stage 3-4 window and full Stage 1-4 window
t_llm, e_llm = ttd_results['mistral']

vuln_static = static_master[static_master['ground_truth'] == 1]
def compute_static_ttd(df, tool_col):
    times, events = [], []
    for sid in df['sample_id'].unique():
        rows = df[df['sample_id'] == sid]
        s3_row = rows[rows['stage'] == 3]
        detected = s3_row.iloc[0][tool_col] == 1 if len(s3_row) > 0 else False
        if detected:
            times.append(3); events.append(1)
        else:
            times.append(4); events.append(0)
    return np.array(times), np.array(events)

t_static, e_static = compute_static_ttd(vuln_static, 'cppcheck_pred')

t_llm_34 = np.where(t_llm < 3, 3, t_llm)
result_34 = logrank_test(t_llm_34, t_static, event_observed_A=e_llm, event_observed_B=e_static)
result_full = logrank_test(t_llm, t_static, event_observed_A=e_llm, event_observed_B=e_static)

print(f"Stage 3 to 4 only:  chi2={result_34.test_statistic:.3f}  p={result_34.p_value:.4g}")
print(f"Full Stage 1 to 4:  chi2={result_full.test_statistic:.3f}  p={result_full.p_value:.4g}")

## 9. Category analysis: best F1 per category, category-versus-rest log-rank (Tables 12, 13, Appendix C)

In [ ]:
categories = sorted(llm_master_v2[llm_master_v2['ground_truth'] == 1]['category'].unique())

def compute_f1_by_stage(df, model_col, category, stage):
    sub = df[(df['stage_name'] == stage) & (df['category'] == category)]
    y_true, y_pred = sub['ground_truth'], sub[model_col]
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    if tp == 0:
        return 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    return 0.0 if (prec + rec == 0) else 2 * prec * rec / (prec + rec)

rows = []
for cat in categories:
    row = {'category': cat}
    for model in llm_models:
        best_f1, best_stage = -1, None
        for stage_num, stage in enumerate(stages, 1):
            f1 = compute_f1_by_stage(llm_master_v2, f'{model}_pred', cat, stage)
            if f1 > best_f1:
                best_f1, best_stage = f1, stage_num
        row[model] = f"{best_f1:.3f} (S{best_stage})"
    rows.append(row)

table12 = pd.DataFrame(rows)
print(table12.to_string(index=False))
table12.to_csv('table12_best_f1_per_category_corrected.csv', index=False)

## 10. Regenerate corrected figures (2, 3, 4, 5)

Figures 6 (EDR) and 7 (CWE category heatmap) were verified unchanged and are not regenerated here. See `figures/` for all final figure files used in the paper.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({'font.family': 'serif', 'font.size': 11, 'axes.linewidth': 0.8,
                      'pdf.fonttype': 42, 'ps.fonttype': 42})

def compute_f1(df, model_col, stage):
    sub = df[df['stage'] == stage]
    tp = ((sub[model_col] == 1) & (sub['ground_truth'] == 1)).sum()
    fp = ((sub[model_col] == 1) & (sub['ground_truth'] == 0)).sum()
    fn = ((sub[model_col] == 0) & (sub['ground_truth'] == 1)).sum()
    if tp == 0:
        return 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return 0.0 if (precision + recall == 0) else 2 * precision * recall / (precision + recall)

llm_master_v2_numeric = llm_master_v2.copy()
llm_master_v2_numeric['stage'] = llm_master_v2_numeric['stage'].astype(int)

f1_matrix = np.zeros((len(llm_models), 4))
for i, model in enumerate(llm_models):
    for j, stage in enumerate([1, 2, 3, 4]):
        f1_matrix[i, j] = compute_f1(llm_master_v2_numeric, f'{model}_pred', stage)

fig, ax = plt.subplots(figsize=(8, 5.5))
im = ax.imshow(f1_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_xticklabels(['Stage 1', 'Stage 2', 'Stage 3', 'Stage 4'])
ax.set_yticks(range(len(llm_models))); ax.set_yticklabels([m.title() for m in llm_models])
for i in range(len(llm_models)):
    for j in range(4):
        ax.text(j, i, f'{f1_matrix[i,j]:.3f}', ha='center', va='center',
                color='white' if f1_matrix[i,j] > 0.55 else 'black')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='F1 Score')
plt.tight_layout()
plt.savefig('figures/FIG2-HEATMAP.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved figures/FIG2-HEATMAP.png")

# See results/ for the full set of corrected tables and figures/ for all
# regenerated plots used in the final paper.

## Summary

This notebook reproduces the correction and every table and figure affected by it: Table 6, 7, 8, 10, 11, 12, 13, Appendix A, B, C, and Figures 2, 3, 4, 5. All outputs match the versions committed to `results/` and `figures/` in this repository, verified line by line against the paper.